# Data manipulation exercise (45 min max)
Your KYC business partner has sent you the data for your project in the `data` folder. 
They want to perform an analysis on the **aggregated incoming transactions** of the clients, in order to see how much  money has been sent to their accounts.

The expected output should look as follows:

`Monika Harrison: 3500.00 EUR`\
`Tina Yu: 2546.96 EUR`\
`Terry Foster: 24895.85 EUR`

In your team there's an abstract data processing class used to process data. Using this abstract class, pre-process the `client` and `transaction` data, and then merge the results to get the desired results.

* `client_db.json` contains client information, such as the email and the primary account of the customer.
* `transaction_db.json` contains transaction information, such as sender account, receiver account and amount transferred.

Clean and process the data to get to the desired output. Explain the steps that you're taking and why. You can consult Google (show your screen) but you cannot use GenAI to solve the problem.

## Step 1 - Plan your activity (5 min max)

Your manager asks you to create a user story (task) with the activity. Fill the information below:

* __User story name__:
* __Expected time completion (min)__:
* __Step by step tasks__:
    - Task 1
    - Task 2

## Step 2 - Develop your solution (40 min  max)
Implement below your solution.

In [1]:
from abc import ABC, abstractmethod
import pandas as pd

class DataProcessing(ABC):
    def __init__(self, path: str):
        self._data_path = path
        self.df = self._load_dataframe(self._data_path)
        
    def _load_dataframe(self, path: str) -> pd.DataFrame:
        # TODO
        return pd.read_json(path)
    
    def preprocess_dataframe(self) -> pd.DataFrame:
        # TODO
        return self.df

    @abstractmethod
    def correct_data_quality(self, df: pd.DataFrame) -> pd.DataFrame:...
    
    @abstractmethod
    def apply_transformations(self, df: pd.DataFrame) -> pd.DataFrame:...

In [50]:
def eur_value_correction(amount):
    if type(amount) is not float:
        try:
            amount = float(amount)
        except:
            amount = 0.0
    return amount

In [60]:
class Client(DataProcessing):
    def correct_data_quality(self, df):
        return df.T.reset_index(drop=True)
    
    def apply_transformations(self, df):
        return df
    
class Transaction(DataProcessing):
    def correct_data_quality(self, df):
        # Check EUR column
        # If empty set it 0.0
        # If it's type is str, convert it to float
        eur_column = df['EUR'].map(eur_value_correction)
        df['EUR'] = eur_column

        return df
        
    
    def apply_transformations(self, df):
        return df

In [61]:
client = Client("data/client_db.json")
client_df = client.df
client_df = client.correct_data_quality(client_df)

transaction = Transaction("data/transaction_db.json")
transaction_df = transaction.df

In [65]:
client_df.head()

,email,account
0,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
1,penelope.langdon@email.com,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0
2,nicola@liame.com,62bdcf27-1f05-4272-a140-a408a8827e0c
3,ella.hardacre@email.com,86c3bf2e-e014-4682-9144-7648775a9a55
4,colin.pike.harris@emaily.com,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c


In [66]:
transaction_df.head()

,acc_in,acc_out,EUR
0,7d55d07b-9bfa-4e28-86fa-28ca925ec985,cea143de-0407-451d-87cd-14cd4723aa8d,3435.41
1,ae629b63-eb36-4f11-94ec-4f61bf7df906,6c19f731-cbf8-4878-94b7-c5557dec5027,2891.44
2,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c,ae629b63-eb36-4f11-94ec-4f61bf7df906,1261.39
3,62bdcf27-1f05-4272-a140-a408a8827e0c,32f52c1a-d1df-433a-b30a-cd34106aa900,999.22
4,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c,62bdcf27-1f05-4272-a140-a408a8827e0c,3661.06


In [67]:
mg = pd.merge(transaction_df, client_df, left_on='acc_in', right_on='account')
mg.head(20)

,acc_in,acc_out,EUR,email,account
0,ae629b63-eb36-4f11-94ec-4f61bf7df906,6c19f731-cbf8-4878-94b7-c5557dec5027,2891.44,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
1,ae629b63-eb36-4f11-94ec-4f61bf7df906,cea143de-0407-451d-87cd-14cd4723aa8d,3412.05,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
2,ae629b63-eb36-4f11-94ec-4f61bf7df906,6c19f731-cbf8-4878-94b7-c5557dec5027,460.82,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
3,ae629b63-eb36-4f11-94ec-4f61bf7df906,cea143de-0407-451d-87cd-14cd4723aa8d,3554.15,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
4,ae629b63-eb36-4f11-94ec-4f61bf7df906,fe48b37c-a92e-4f6e-9e87-3036b717ef00,485.73,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
5,ae629b63-eb36-4f11-94ec-4f61bf7df906,74d62c94-a044-465a-83ea-63d673527bd3,0.00,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
6,ae629b63-eb36-4f11-94ec-4f61bf7df906,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0,2705.33,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
7,ae629b63-eb36-4f11-94ec-4f61bf7df906,6c19f731-cbf8-4878-94b7-c5557dec5027,3852.24,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
8,ae629b63-eb36-4f11-94ec-4f61bf7df906,74d62c94-a044-465a-83ea-63d673527bd3,3597.67,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
9,ae629b63-eb36-4f11-94ec-4f61bf7df906,4177f6a4-f43b-4c4f-95a0-3391fc94aed5,1896.55,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906


In [68]:
mg = pd.merge(mg, client_df, left_on='acc_out', right_on='account')

In [69]:
mg.head(20)

,acc_in,acc_out,EUR,email_x,account_x,email_y,account_y
0,ae629b63-eb36-4f11-94ec-4f61bf7df906,cea143de-0407-451d-87cd-14cd4723aa8d,3412.05,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906,evan@mathis@liame.com,cea143de-0407-451d-87cd-14cd4723aa8d
1,ae629b63-eb36-4f11-94ec-4f61bf7df906,cea143de-0407-451d-87cd-14cd4723aa8d,3554.15,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906,evan@mathis@liame.com,cea143de-0407-451d-87cd-14cd4723aa8d
2,62bdcf27-1f05-4272-a140-a408a8827e0c,cea143de-0407-451d-87cd-14cd4723aa8d,4524.02,nicola@liame.com,62bdcf27-1f05-4272-a140-a408a8827e0c,evan@mathis@liame.com,cea143de-0407-451d-87cd-14cd4723aa8d
3,ae629b63-eb36-4f11-94ec-4f61bf7df906,fe48b37c-a92e-4f6e-9e87-3036b717ef00,485.73,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
4,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c,fe48b37c-a92e-4f6e-9e87-3036b717ef00,1640.67,colin.pike.harris@emaily.com,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
5,74d62c94-a044-465a-83ea-63d673527bd3,fe48b37c-a92e-4f6e-9e87-3036b717ef00,3518.63,jasmine.paige@email.com,74d62c94-a044-465a-83ea-63d673527bd3,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
6,74d62c94-a044-465a-83ea-63d673527bd3,fe48b37c-a92e-4f6e-9e87-3036b717ef00,4308.91,jasmine.paige@email.com,74d62c94-a044-465a-83ea-63d673527bd3,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
7,74d62c94-a044-465a-83ea-63d673527bd3,fe48b37c-a92e-4f6e-9e87-3036b717ef00,3012.67,jasmine.paige@email.com,74d62c94-a044-465a-83ea-63d673527bd3,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
8,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0,fe48b37c-a92e-4f6e-9e87-3036b717ef00,3113.36,penelope.langdon@email.com,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
9,ae629b63-eb36-4f11-94ec-4f61bf7df906,74d62c94-a044-465a-83ea-63d673527bd3,0.00,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906,jasmine.paige@email.com,74d62c94-a044-465a-83ea-63d673527bd3


In [70]:
unique_acc_in = pd.unique(mg["acc_in"])
unique_acc_out = pd.unique(mg["acc_out"])

print(len(unique_acc_in), len(unique_acc_out))

8 8


In [73]:
in_df = mg.groupby(by=['acc_in'])["EUR"].sum()
in_df

acc_in
0cdcfe55-e99f-45b9-af9d-039a2e5e51b5    1.510379e+07
62bdcf27-1f05-4272-a140-a408a8827e0c    1.085373e+04
74d62c94-a044-465a-83ea-63d673527bd3    1.442143e+04
75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c    2.318224e+04
ae629b63-eb36-4f11-94ec-4f61bf7df906    2.050190e+04
d4c094cd-f1e2-4e95-b686-c6b28fd43dd8    8.775290e+03
d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0    9.097590e+03
fe48b37c-a92e-4f6e-9e87-3036b717ef00    5.017818e+06
Name: EUR, dtype: float64

In [74]:
out_df = mg.groupby(by=['acc_out'])["EUR"].sum()
out_df

acc_out
0cdcfe55-e99f-45b9-af9d-039a2e5e51b5    1.139320e+04
62bdcf27-1f05-4272-a140-a408a8827e0c    2.410017e+04
74d62c94-a044-465a-83ea-63d673527bd3    1.474425e+04
86c3bf2e-e014-4682-9144-7648775a9a55    5.011332e+06
ae629b63-eb36-4f11-94ec-4f61bf7df906    1.510993e+07
cea143de-0407-451d-87cd-14cd4723aa8d    1.149022e+04
d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0    9.370720e+03
fe48b37c-a92e-4f6e-9e87-3036b717ef00    1.607997e+04
Name: EUR, dtype: float64

In [84]:
df = pd.DataFrame({
    'incoming': in_df,
    'outgoing': out_df
}).fillna(0.0)

# 2) Reset index → makes a new column called 'acc_in' (the old index name)
df = df.reset_index()

# 3) (Optional) rename that column to something friendlier
df = df.rename(columns={'acc_in': 'account_id'})

# 4) Compute net / final amount
df['final_amount'] = df['incoming'] - df['outgoing']


df.head(20)

,index,incoming,outgoing,final_amount
0,0cdcfe55-e99f-45b9-af9d-039a2e5e51b5,1.510379e+07,1.139320e+04,1.509240e+07
1,62bdcf27-1f05-4272-a140-a408a8827e0c,1.085373e+04,2.410017e+04,-1.324644e+04
2,74d62c94-a044-465a-83ea-63d673527bd3,1.442143e+04,1.474425e+04,-3.228200e+02
3,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c,2.318224e+04,0.000000e+00,2.318224e+04
4,86c3bf2e-e014-4682-9144-7648775a9a55,0.000000e+00,5.011332e+06,-5.011332e+06
5,ae629b63-eb36-4f11-94ec-4f61bf7df906,2.050190e+04,1.510993e+07,-1.508943e+07
6,cea143de-0407-451d-87cd-14cd4723aa8d,0.000000e+00,1.149022e+04,-1.149022e+04
7,d4c094cd-f1e2-4e95-b686-c6b28fd43dd8,8.775290e+03,0.000000e+00,8.775290e+03
8,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0,9.097590e+03,9.370720e+03,-2.731300e+02
9,fe48b37c-a92e-4f6e-9e87-3036b717ef00,5.017818e+06,1.607997e+04,5.001738e+06


In [85]:
mg_df = pd.merge(df, client_df, left_on='index', right_on='account')
mg_df.head(20)

,index,incoming,outgoing,final_amount,email,account
0,0cdcfe55-e99f-45b9-af9d-039a2e5e51b5,1.510379e+07,1.139320e+04,1.509240e+07,.nolan@email.com,0cdcfe55-e99f-45b9-af9d-039a2e5e51b5
1,62bdcf27-1f05-4272-a140-a408a8827e0c,1.085373e+04,2.410017e+04,-1.324644e+04,nicola@liame.com,62bdcf27-1f05-4272-a140-a408a8827e0c
2,74d62c94-a044-465a-83ea-63d673527bd3,1.442143e+04,1.474425e+04,-3.228200e+02,jasmine.paige@email.com,74d62c94-a044-465a-83ea-63d673527bd3
3,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c,2.318224e+04,0.000000e+00,2.318224e+04,colin.pike.harris@emaily.com,75be85c0-0f5f-4bfb-a2dc-0b2f39e63d3c
4,86c3bf2e-e014-4682-9144-7648775a9a55,0.000000e+00,5.011332e+06,-5.011332e+06,ella.hardacre@email.com,86c3bf2e-e014-4682-9144-7648775a9a55
5,ae629b63-eb36-4f11-94ec-4f61bf7df906,2.050190e+04,1.510993e+07,-1.508943e+07,samantha.gibson@email.com,ae629b63-eb36-4f11-94ec-4f61bf7df906
6,cea143de-0407-451d-87cd-14cd4723aa8d,0.000000e+00,1.149022e+04,-1.149022e+04,evan@mathis@liame.com,cea143de-0407-451d-87cd-14cd4723aa8d
7,d4c094cd-f1e2-4e95-b686-c6b28fd43dd8,8.775290e+03,0.000000e+00,8.775290e+03,vanessa.thomson@email.com,d4c094cd-f1e2-4e95-b686-c6b28fd43dd8
8,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0,9.097590e+03,9.370720e+03,-2.731300e+02,penelope.langdon@email.com,d708f0e0-1879-4fc0-9bb3-a136b7e6f3a0
9,fe48b37c-a92e-4f6e-9e87-3036b717ef00,5.017818e+06,1.607997e+04,5.001738e+06,edward.oliver@email.com,fe48b37c-a92e-4f6e-9e87-3036b717ef00
